###### Contact Lens Wettability Assay: Validation Study

<details>
    <summary><b>List of Abberviations</b> <i>(click to expand)</i></summary>
    RT: Room Temperature<br>
    OT: Ocular Temperaute (37C)<br>
    PP_holder: Pipette-Pinion holder<br>
    CA: Contact Angle
</details>

## Step 1: Imports + Set Home

In [2]:
# Rerun if it initially errors
from opentrons import protocol_api
import socket
from opentrons.types import *
import opentrons.execute

protocol = opentrons.execute.get_protocol_api("2.19")
protocol.home()

Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 402, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 415, in read_labware_latch
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 154, in get_labware_latch_status
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: 

#### Run in Anaconda Prompt:
conda activate RAISE-Vision\
cd Desktop/RAISE-Vision\
python CameraServer.py

## Step 2: Defining Labware

In [3]:
#Pipettes
left = protocol.load_instrument("p20_single_gen2", "left")             # uses "dryer tips" and "tips" for tipracks
right = protocol.load_instrument('p1000_single_gen2', 'right')         # won't use tip racks

#Liquid Handling
dryer_tips = protocol.load_labware("opentrons_96_tiprack_20ul", 10) 
tips = protocol.load_labware("opentrons_96_tiprack_20ul", 11) 
well = protocol.load_labware("corning_24_wellplate_3.4ml_flat", 5)     # sessile drop liquid 24-well plate

#Custom Labware
stage = protocol.load_labware('fgl_stage', 4)                          # use 1 for RT, use 4 for OT
drypad = protocol.load_labware("fgl_drypad_tiprack", 6)
drypad2 = protocol.load_labware("fgl_drypad_tiprack", 3)
PP_holder = protocol.load_labware('fgl_1_tiprack_1000ul', 7)           # PP_holder: Pipette-Pinion holder
rack_lens = protocol.load_labware('fgl_lens', 8)
trash = protocol.load_labware("fgl_trash", 9)                          # trash for dry tips
#gear = (need to define & fix offset, currently just using offset)

Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 401, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 412, in read_rpm
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 141, in get_rpm
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response to 'M123 
'


In [ ]:
# large_drypad = protocol.load_labware("fgl_single_drypad_tiprack", 2)

In [ ]:
# #pick up drying apparatus
# loc = large_drypad.wells()[0]
# left.pick_up_tip(loc)

# #return drying apparatus
# left.return_tip()

## Step 3: Camera Set-Up

In [4]:
#Define Camera Class
class Camera:
    def __init__(self, ip_address: str, port: int):
        self.client_socket = socket.create_connection((ip_address, port))
    
    def take_image(self, fname: str) -> None:
        msg: str = f"take-image,{fname}"
        self.client_socket.send(msg.encode())
        self.client_socket.recv(1024)
        
    def new_folder(self, dir_name: str) -> None:
        msg: str = f"new-folder,{dir_name}"
        self.client_socket.send(msg.encode())
        self.client_socket.recv(1024)
    
    def analyze_images(self, dir_name: str) -> None:
        msg: str = f"analyze-images,{dir_name}"
        self.client_socket.send(msg.encode())
        self.client_socket.recv(1024)
    
    def __enter__(self):
        return self
    
    def close(self):
        self.client_socket.close()
    
    def __exit__(self, exc_type, exc_val, traceback):
        self.client_socket.close()


#### Remember to Activate Camera in Anaconda Prompt
python CameraServer.py

In [5]:
#Initiate Camera Connection
from time import time
IP_ADDRESS = "192.168.0.111"
PORT = 1200
cam = Camera(IP_ADDRESS, PORT)

## Step 4. Define Lens Testing Functions

In [6]:
def drySides(rolledDryerLoc):
    #Standardized initial drying protocol using KimWipe insert
    #Wicks away excess moisture from contact lens

    high = 84.5
    low = 40.5 #40.5
    
    large_offset = 11
    small_offset = 5.5 #4.5, 5.5
    x_offset = -1.5
    fastSpeed = 70
    slowSpeed = 3
    waitTime = 3


    #pick up from defined well
    dryerLoc = dryer_tips.wells()[rolledDryerLoc]
    left.pick_up_tip(dryerLoc)

    #go to Stage
    loc = stage.wells_by_name()['B2']

    #dry back side
    stage.set_offset(x=x_offset+0, y=large_offset, z=0)
    left.move_to(loc.top(high), force_direct = False, speed = fastSpeed)
    left.move_to(loc.top(low), force_direct = False, speed = fastSpeed)

    stage.set_offset(x=x_offset+0, y=small_offset, z=0)
    left.move_to(loc.top(low), force_direct = False, speed = slowSpeed)
    protocol.delay(waitTime)

    stage.set_offset(x=x_offset+0, y=large_offset, z=0)
    left.move_to(loc.top(low), force_direct = False, speed = slowSpeed)
    left.move_to(loc.top(high), force_direct = False, speed = fastSpeed)

    #dry left side
    stage.set_offset(x=-large_offset, y=0, z=0)
    left.move_to(loc.top(high), force_direct = False, speed = fastSpeed)
    left.move_to(loc.top(low), force_direct = False, speed = fastSpeed)

    stage.set_offset(x=-small_offset, y=0, z=0)
    left.move_to(loc.top(low), force_direct = False, speed = slowSpeed)
    protocol.delay(waitTime)

    stage.set_offset(x=-large_offset, y=0, z=0)
    left.move_to(loc.top(low), force_direct = False, speed = slowSpeed)
    left.move_to(loc.top(high), force_direct = False, speed = fastSpeed)

    #dry front side
    stage.set_offset(x=0, y=-large_offset, z=0)
    left.move_to(loc.top(high), force_direct = False, speed = fastSpeed)
    left.move_to(loc.top(low), force_direct = False, speed = fastSpeed)

    stage.set_offset(x=0, y=-small_offset, z=0)
    left.move_to(loc.top(low), force_direct = False, speed = slowSpeed)
    protocol.delay(waitTime)

    stage.set_offset(x=0, y=-large_offset, z=0)
    left.move_to(loc.top(low), force_direct = False, speed = slowSpeed)
    left.move_to(loc.top(high), force_direct = False, speed = fastSpeed)
    
    #dry right side
    stage.set_offset(x=x_offset+large_offset, y=0, z=0)
    left.move_to(loc.top(high), force_direct = False, speed = fastSpeed)
    left.move_to(loc.top(low), force_direct = False, speed = fastSpeed)

    stage.set_offset(x=x_offset+small_offset, y=0, z=0)
    left.move_to(loc.top(low), force_direct = False, speed = slowSpeed)
    protocol.delay(waitTime)

    stage.set_offset(x=x_offset+large_offset, y=0, z=0)
    left.move_to(loc.top(low), force_direct = False, speed = slowSpeed)
    left.move_to(loc.top(high), force_direct = False, speed = fastSpeed)

    #drop tip in defined trash
    throw = trash.wells()[1]
    left.drop_tip(throw)
    

def dryDroplet(wellNum): 
    #Remove CA droplet from contact lens using KimWipe

    #pick up drying apparatus
    loc = drypad.wells()[wellNum]
    left.pick_up_tip(loc)

    #dry the actual droplet
    stage.set_offset(x=62.5, y=0.65, z=-8.00) #y = 0.65 works better than y = 0 (checked drypads)
    loc = stage.wells_by_name()['B1']
    
    left.move_to(loc.top(100), force_direct = False)
    left.move_to(loc.top(72), force_direct = False, speed = 70)
    left.move_to(loc.top(66.75), force_direct = False, speed = 2) #47.6, 61.85
    protocol.delay(3)
    left.move_to(loc.top(100), force_direct = False)

    #return drying apparatus
    left.return_tip()

def dryDroplet2(wellNum): #for drypad2
    #Remove CA droplet from contact lens using KimWipe

    #pick up drying apparatus
    loc = drypad2.wells()[wellNum]
    left.pick_up_tip(loc)

    #dry the actual droplet
    stage.set_offset(x=62.5, y=0.65, z=-8.00) #y = 0.65 works better than y = 0 (checked drypads)
    loc = stage.wells_by_name()['B1']
    
    left.move_to(loc.top(100), force_direct = False)
    left.move_to(loc.top(72), force_direct = False, speed = 70)
    left.move_to(loc.top(66.75), force_direct = False, speed = 2) #47.6, 61.85
    protocol.delay(3)
    left.move_to(loc.top(100), force_direct = False)

    #return drying apparatus
    left.return_tip()


def droplet(i, incubationNum, CycleNum, run_dir, blink):
    stage.set_offset(x=62.5, y=-0.25, z=-10.00)
    stg = stage.wells()[1]
    
    #Locations 
    locA = stg.top(74.5)
    locB = stg.top(72.5)

    #ASPIRATE
    pip = tips.wells()[i+incubationNum*CycleNum]
    left.pick_up_tip(pip)
    
    #Pre-lens Image
    if blink == False:
        fname = run_dir + f"preLens_run" + str(incubationNum+1) + "_drop" + str(i+1) + ".jpg" #BEFORE: f"preLens_cycle" + str(i+1) + "_drop" + str(incubationNum+1) + ".jpg"
    else:
        fname = run_dir + f"preLens_run1_cycle" + str(incubationNum+1) + ".jpg" #b/c 5 drops is actually 5 cycles (with soak in between)
    cam.take_image(fname)
    
    left.aspirate(5, well['B3']) #B3 for PBS. A1 for MQ

    # CREATE DROPLET ON PIPETTE TIP     
    left.move_to(locA, force_direct = False)
    protocol.delay(1)
    left.dispense(5, locA, rate = 1) # Dispense 5 uL
    left.move_to(locB, force_direct = True, speed = 0.5)
    
    #Safety Factor
    left.move_to(locA, force_direct = True, speed = 2)
    
    #VIEW DROPLET + TAKE IMAGE   
    protocol.delay(5)
    if blink == False:
        fname = run_dir + f"Image_run" + str(incubationNum+1) + "_drop" + str(i+1) + ".jpg"
    else:
        fname = run_dir + f"Image_run1_cycle" + str(incubationNum+1) + ".jpg"
    cam.take_image(fname)
    protocol.delay(1)
    
    #Drop the tip in the TRASH
    left.drop_tip() 

def single_droplet(i, tipnNum, incubationNum, CycleNum, run_dir, well_loc): #just made a new code to edit the well number easily
    stage.set_offset(x=62.5, y=-0.25, z=-10.00)
    stg = stage.wells()[1]
    
    #Locations 
    locA = stg.top(74.5)
    locB = stg.top(72.5)

    #ASPIRATE
    pip = tips.wells()[i+tipnNum*CycleNum]
    left.pick_up_tip(pip)
    
    #Pre-lens Image
    fname = run_dir + f"preLens_run1_cycle" + str(incubationNum+1) + ".jpg" #b/c 5 drops is actually 5 cycles (with soak in between)
    cam.take_image(fname)
    
    left.aspirate(5, well[well_loc]) #B6 for PBS. A1 for MQ #'B1'

    # CREATE DROPLET ON PIPETTE TIP     
    left.move_to(locA, force_direct = False)
    protocol.delay(1)
    left.dispense(5, locA, rate = 1) # Dispense 5 uL
    left.move_to(locB, force_direct = True, speed = 0.5)
    
    #Safety Factor
    left.move_to(locA, force_direct = True, speed = 2)
    
    #VIEW DROPLET + TAKE IMAGE   
    protocol.delay(5)
    fname = run_dir + f"Image_run1_cycle" + str(incubationNum+1) + ".jpg"
    cam.take_image(fname)
    protocol.delay(1)
    
    #Drop the tip in the TRASH
    left.drop_tip() 

In [ ]:
# a = 10

In [ ]:
# drySides(a)
# a += 1

In [ ]:
# i = 0
# a1 = 10

In [ ]:
# left.drop_tip() 

In [ ]:
# dropletNum = 1
# run_dir = '../HELLO/'
# droplet(0, a1, dropletNum, run_dir, True)

# a1 += 1

In [ ]:
#b = 0

In [ ]:
# dryDroplet(b)
# b += 1

## Step 5. Substrate (Lens) Handling and Rotation Functions

In [7]:
#Pick-and-Place Function for the pipette-pinion assembly.
def gearPickUp():
    loc = PP_holder.wells()[0]
    right.pick_up_tip(loc)
    
def gearDropOff():
    loc = PP_holder.wells()[0]    
    right.drop_tip(loc)

    
#Rotation Function - engages pinion teeth with clip-on rack
def flip():
    #Define well locations to guide rotation movement.
    #These locations can be customized based on the hardware you are using to insert the rack.
    #The clip-on rack is inserted at the deep well plate between wells C2 and C5.
    locA = rack_lens.wells_by_name()['C3']
    locB = rack_lens.wells_by_name()['C5']

    #move to rack & engage teeth
    rack_lens.set_offset(x=0.5, y=-20, z=0.00)
    right.move_to(locA.top(30), force_direct = False, speed = 250)
    right.move_to(locA.top(7.5), force_direct = False, speed = 250)

    #move across rack
    right.move_to(locB.top(7.5),force_direct = False, speed = 100)
    right.move_to(locB.top(30), force_direct = False, speed = 250)
    
#Stage Drop-off and Pick-up
def substratePickUp():
    loc = stage.wells_by_name()['B2']

    #move to correct z for substrate holder extension
    stage.set_offset(x=-1.5, y=-62, z=-0.25)
    right.move_to(loc.top(70), force_direct = False, speed = 60)
    right.move_to(loc.top(32), force_direct = False, speed = 30)

    #move laterally (towards) to pick up substrate holder
    stage.set_offset(x=-1.5, y=-50, z=-0.25)
    right.move_to(loc.top(32), force_direct = False, speed = 10)
    right.move_to(loc.top(70), force_direct = False, speed = 100) #moves up after picking up mushroom
    
def substrateDropOff():
    loc = stage.wells_by_name()['B2']

    stage.set_offset(x=-1.5, y=-50.5, z=-0.25)
    right.move_to(loc.top(70), force_direct = False, speed = 250)
    right.move_to(loc.top(32), force_direct = False, speed = 250)

    stage.set_offset(x=-1.5, y=-62, z=-0.25)
    right.move_to(loc.top(32), force_direct = False, speed = 250)
    
    stage.set_offset(x=-1.5, y=-62, z=30)
    right.move_to(loc.top(70), force_direct = False, speed = 70)
    
#Lens Pickup and Dropoff
def lensPickUp(lensLoc):
    loc = rack_lens.wells_by_name()[lensLoc] #Can be D1, D2, D3, D4, D5, D6

    #move to lens well
    rack_lens.set_offset(x=0, y=-49.5, z=0)
    right.move_to(loc.top(75), force_direct = False, speed = 100)
    right.move_to(loc.top(10), force_direct = False, speed = 80)

    #dip lens mount into well
    right.move_to(loc.top(-5), force_direct = False, speed = 5)
    protocol.delay(1.5)

    #lift lens out of well
    right.move_to(loc.top(10), force_direct = False, speed = 5)
    right.move_to(loc.top(75), force_direct = False, speed = 100)
    
    
def lensDropOff(lensLoc):
    loc =  rack_lens.wells_by_name()[lensLoc] #Can be D1, D2, D3, D4, D5, D6

    #move to lens well
    rack_lens.set_offset(x=0, y=-49.5, z=0)
    right.move_to(loc.top(75), force_direct = False, speed = 100)
    right.move_to(loc.top(10), force_direct = False, speed = 80)

    #dip lens mount into well
    right.move_to(loc.top(-2), force_direct = False, speed = 5)
    protocol.delay(1.5)

    #dislodge lens from well by moving in y axis
    rack_lens.set_offset(x=0, y=-47, z=0)
    right.move_to(loc.top(-2), force_direct = False, speed = 5)
    protocol.delay(1.5)

    #lift lens mount out of well
    right.move_to(loc.top(10), force_direct = False, speed = 50)
    right.move_to(loc.top(75), force_direct = False, speed = 100)

def lensSoak(lensLoc, soakTime):
    loc =  rack_lens.wells_by_name()[lensLoc] #Can be D1, D2, D3, D4, D5, D6

    #move to lens well
    rack_lens.set_offset(x=0, y=-49.5, z=0)
    right.move_to(loc.top(75), force_direct = False, speed = 100)
    right.move_to(loc.top(10), force_direct = False, speed = 80)

    #dip lens mount into well
    right.move_to(loc.top(0), force_direct = False, speed = 5)
    print(f"Delaying for {soakTime} seconds.")
    protocol.delay(soakTime)

    #lift lens mount out of well
    right.move_to(loc.top(10), force_direct = False, speed = 5)
    right.move_to(loc.top(75), force_direct = False, speed = 100)

def lensSoak_andPickUp(lensLoc, soakTime):
    loc =  rack_lens.wells_by_name()[lensLoc] #Can be D1, D2, D3, D4, D5, D6

    #move to lens well
    rack_lens.set_offset(x=0, y=-49.5, z=0)
    right.move_to(loc.top(75), force_direct = False, speed = 100)
    right.move_to(loc.top(10), force_direct = False, speed = 80)

    #dip lens mount into well
    right.move_to(loc.top(0), force_direct = False, speed = 5)
    print(f"Delaying for {soakTime} seconds.")
    protocol.delay(soakTime)

    #dip lens mount further into well to pick up lens in case it fell
    right.move_to(loc.top(-2), force_direct = False, speed = 5)
    protocol.delay(1)

    #lift lens mount out of well
    right.move_to(loc.top(10), force_direct = False, speed = 5) #also changed to 5 instead of 50
    right.move_to(loc.top(75), force_direct = False, speed = 100)

In [ ]:
# substratePickUp()
# flip()
# lensSoak('D1', 4)
# flip()
# substrateDropOff()

In [38]:
# #for pronged lens scoop
# def lensSoak2(lensLoc, soakTime):
#     loc =  rack_lens.wells_by_name()[lensLoc] #Can be D1, D2, D3, D4, D5, D6

#     #move to lens well from centre
#     rack_lens.set_offset(x=0, y=-49.5, z=0)
#     right.move_to(loc.top(75), force_direct = False, speed = 100)
#     right.move_to(loc.top(5), force_direct = False, speed = 80)
    
#     #dip lens mount into well while moving lens to left side to avoid the prongs
#     rack_lens.set_offset(x=-1, y=-49.5, z=0)
#     right.move_to(loc.top(0), force_direct = False, speed = 5)
#     print(f"Delaying for {soakTime} seconds.")
#     protocol.delay(soakTime)
    
#     #perform a side-to-side shake to try to get water underneath lens (also moved to z = 0 to prevent wrecking lens)
#     for i in range(5):
#         for i in range(10):
#             rack_lens.set_offset(x=-1.5, y=-49.5, z=-0.75)
#             right.move_to(loc.top(0), force_direct = False, speed = 25)
#             rack_lens.set_offset(x=1.5, y=-49.5, z=-0.75)
#             right.move_to(loc.top(0), force_direct = False, speed = 25)
# #         for i in range(10):
# #             rack_lens.set_offset(x=1.25, y=-50.5, z=0)
# #             right.move_to(loc.top(0), force_direct = False, speed = 30)
# #             rack_lens.set_offset(x=1.25, y=-50.5, z=3.5)
# #             right.move_to(loc.top(0), force_direct = False, speed = 30)
#         for i in range(10):
#             rack_lens.set_offset(x=0, y=-50, z=-0.75)
#             right.move_to(loc.top(0), force_direct = False, speed = 25)
#             rack_lens.set_offset(x=0, y=-49, z=-0.75)
#             right.move_to(loc.top(0), force_direct = False, speed = 25)


    
#     #move lens to right side & slower lift it up
#     rack_lens.set_offset(x=1.5, y=-49.5, z=-0.5)
#     right.move_to(loc.top(1), force_direct = False, speed = 2)
#     rack_lens.set_offset(x=2, y=-49.5, z=0)
#     right.move_to(loc.top(5), force_direct = False, speed = 2)

#     #lift lens mount out of well
#     right.move_to(loc.top(10), force_direct = False, speed = 5)
#     right.move_to(loc.top(75), force_direct = False, speed = 100)
    
# substratePickUp()
# flip()
# lensSoak2('D6', 10)
# flip()
# substrateDropOff()

Delaying for 10 seconds.


In [22]:
# #for pronged lens scoop
# def lensSoak2(lensLoc, soakTime):
#     loc =  rack_lens.wells_by_name()[lensLoc] #Can be D1, D2, D3, D4, D5, D6

#     #move to lens well from centre
#     rack_lens.set_offset(x=0, y=-49.5, z=0)
#     right.move_to(loc.top(75), force_direct = False, speed = 100)
#     right.move_to(loc.top(5), force_direct = False, speed = 80)
    
#     #dip lens mount into well while moving lens to left side to avoid the prongs
#     rack_lens.set_offset(x=-1, y=-49.5, z=0)
#     right.move_to(loc.top(0), force_direct = False, speed = 5)
#     print(f"Delaying for {soakTime} seconds.")
#     protocol.delay(soakTime)
    
#     #perform a side-to-side shake to try to get water underneath lens (also moved to z = 0 to prevent wrecking lens)
#     for i in range(5):
#         for i in range(10):
#             rack_lens.set_offset(x=-1.5, y=-49.5, z=-0.75)
#             right.move_to(loc.top(0), force_direct = False, speed = 25)
#             rack_lens.set_offset(x=1.5, y=-49.5, z=-0.75)
#             right.move_to(loc.top(0), force_direct = False, speed = 25)
# #         for i in range(10):
# #             rack_lens.set_offset(x=1.25, y=-50.5, z=0)
# #             right.move_to(loc.top(0), force_direct = False, speed = 30)
# #             rack_lens.set_offset(x=1.25, y=-50.5, z=3.5)
# #             right.move_to(loc.top(0), force_direct = False, speed = 30)
#         for i in range(10):
#             rack_lens.set_offset(x=0, y=-50, z=-0.75)
#             right.move_to(loc.top(0), force_direct = False, speed = 25)
#             rack_lens.set_offset(x=0, y=-49, z=-0.75)
#             right.move_to(loc.top(0), force_direct = False, speed = 25)


    
#     #move lens to right side & slower lift it up
#     rack_lens.set_offset(x=1.5, y=-49.5, z=-0.5)
#     right.move_to(loc.top(1), force_direct = False, speed = 2)
#     rack_lens.set_offset(x=2, y=-49.5, z=0)
#     right.move_to(loc.top(5), force_direct = False, speed = 2)

#     #lift lens mount out of well
#     right.move_to(loc.top(10), force_direct = False, speed = 5)
#     right.move_to(loc.top(75), force_direct = False, speed = 100)
    
# substratePickUp()
# flip()
# lensSoak2('B5', 10)
# flip()
# substrateDropOff()

In [ ]:
# substratePickUp()
# flip()
# lensPickUp('D1')
# flip()
# substrateDropOff()
# protocol.delay(5)
# drySides()

## Step 6. Protocol Variables: Version 2 (@ 37°C)

In [8]:
#Constants
seconds = 60                                        #60 seconds per minute
drySides_time = 54                                  #54 s for dry sides
droplet_time = 17                                   #17 s for droplet to be placed

#Protocol delay definitions
OT2initial_pickLens_airTime = 21                    #Amount of air exposure of lens within initial pick up protocol
OT2initial_pickLens_soakTime = 31                   #Amount of extra lens soaking time within the initial pick up protocol
OT2end_dropLens_soakTime = 20                       #Amount of extra lens soaking time within the ending drop off protocol
OT2middle_missingHeatedPlate_time = 15.5            #Amount of time on the heated plate misssed in the Runs #(>1) (for Runs 2+)

<b>Protocol Delay Variables Descriptions:</b>
- OT2<b>initial</b>_pickLens_airTime = lens out of soln within lensPickUp() → right before drySides()
- OT2<b>initial</b>_pickLens_soakTime = substratePickUp() → lens out of soln within lensPickUp()
- OT2<b>end</b>_dropLens_soakTime = lens drop in soln within lensDropOff() → end of substrateDropOff()
- OT2<b>middle</b>_missingHeatedPlate_time = calculated time of substrate on heated plate that is missing from Runs 2+ compared to Run 1

Notes:\
• airTime = air exposure time\
• soakTime = soak in solution time\
• initial = before a Run ("fresh" lens getting picked up)\
• end = after a Run (tested lens returning to solution)
• middle = occurs during a Run

## Step 7. Experiment: Version 2 (@ 37°C)

### <b>Rewetting</b> Contact Lens Wetting Protocol

<span style="color: blue;"><b>CHECKLIST:</b><br>
- is the camera on?<br>
- are the drypads placed in the correct orientation?<br>
- does user place lens first?<br>
- set <code style="color: blue;">soakTime</code> to desired value!<br>
- check # of incubations/runs (<code style="color: blue;">incubationCycleNum</code>) and # of sessile drops (<code style="color: blue;">dropletNum</code>)</span>

#### Remember to Activate Camera in Anaconda Prompt
python CameraServer.py

In [17]:
#Initiate Camera Connection
from time import time
IP_ADDRESS = "192.168.0.111"
PORT = 1200
cam = Camera(IP_ADDRESS, PORT)

Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 401, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 412, in read_rpm
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 141, in get_rpm
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response to 'M123 
'
Pol

In [9]:
#Parameters
userPlaceLensFirst = True                           #True if user places the lens manually first
initial_dry_time = 5.5*seconds                      #Blink protocol calls for 1.5 minutes of initial drying (@ 37°C)
drop1_to_drop5_time = 240                           #Time after drop 1 to before drop 5 (is 4 min)

#Protocol delay definitions
OT2initial_autoRun1_airTime = 316.5                 #Amount of air time before droplet for first Run1 with userPlaceLens = False

#Loop each cycle (drypad location is incubation cycle dependent)
incubationCycleNum = 4                              #Number of time lens is to be incubated: 4 times
soakTime = 314                                      #in s; soak time for Rewetting part
dropletNum = 5                                      #Total sessile drop times; should be 1 time for Blinking

In [ ]:
## LONG TRIAL TEST
incubationCycleNum = 16                              #Lens to be incubated 16 times (max possible for 1 full rack of pipette tips)

In [10]:
#Pre-Experiment Setup
gearPickUp()

In [11]:
#a_dry = 0

In [16]:
#Initialize Specific Variables
b = 0

#Create Folder Directory for this Trial in Desktop
run_dir = '../20260807-senofilcon AOH (2030)-4p1.5m-RV2Rewett-37-PBSd-0.9wv%NaCls (2.1)/' #PBSd-1vv%ED(U)s, 1wv%PEG400s
print("Created File Directory")
cam.new_folder(run_dir)

#Looping through the Incubation Cycles
for a in range(incubationCycleNum):
    if (userPlaceLensFirst and a == 0):
        #run initial drying protocol
        drySides(a_dry) 
        a_dry += 1
        if (a_dry >= 96):
            a_dry = 0

        #delay for remaining time
        protocol.delay(initial_dry_time - drySides_time - droplet_time - drop1_to_drop5_time) #is 330 - 54 - 17 - 240 = 26 s 

    else:
        #run Pickup/Dropoff Lens protocol: well -> heated plate
        substratePickUp()
        flip()
        if a == 0: #if first cycle
            lensPickUp('D1')
            flip()
            protocol.delay(initial_dry_time - OT2initial_autoRun1_airTime) #is 330 - 316.5 = 13.5 s, after flip() so substrate it upright (prevents lens from falling)
        else: #o/w, soak lens first before next cycle
            lensSoak_andPickUp('D1', soakTime)
            flip()
        substrateDropOff()

        #run initial drying protocol
        drySides(a_dry)
        a_dry += 1
        if (a_dry >= 96):
            a_dry = 0

        #delay for enough time on heated mount
        protocol.delay(OT2middle_missingHeatedPlate_time) #is 15.5 s
    
    ## Contact Angle Measurement
    for i in range(dropletNum):
        #place droplet of MQ (or droplet solution) on contact lens
        droplet(i, a, dropletNum, run_dir, False)
        
#         fname = run_dir + f"Image_run" + str(a+1) + "_drop" + str(i+1) + "_two.jpg"
#         cam.take_image(fname)

        #dry droplet from contact lens
        if b <= 9:
            dryDroplet(b)
        else:
            b2 = b - 10
            dryDroplet2(b2)

        #adjust drypad pickup to next-in-line (for next cycle)
        b += 1
#         if (b >= 10): #10 b/c it's after 2 full trials
#             b = 0
        
#run Pickup/Dropoff Lens protocol: heated plate -> well
substratePickUp()   
flip()
lensDropOff('D1')
flip()
substrateDropOff()
    
#run image analysis
print("Running Image Analysis")
cam.analyze_images(run_dir)

Created File Directory


Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/utils.py", line 62, in parse_temperature_response
KeyError: 'T'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 400, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 409, in read_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 123, in get_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/utils.py", line 65, in parse_temperature_response
opentrons.drivers.utils.ParseError: ParseError(error_message=Unexpected argument to parse_temperature_response, parse_source=TUS:IDLE_CLOSED)
Polling

Delaying for 314 seconds.


Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 400, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 409, in read_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 120, in get_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response

Delaying for 314 seconds.


Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 401, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 412, in read_rpm
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 141, in get_rpm
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response to 'M123 
'
Pol

Delaying for 314 seconds.


Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 400, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 409, in read_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 120, in get_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response

Running Image Analysis


Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 401, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 412, in read_rpm
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 141, in get_rpm
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response to 'M123 
'


In [18]:
#Post-Experiment
gearDropOff()

Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 402, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 415, in read_labware_latch
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 154, in get_labware_latch_status
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: 

KeyboardInterrupt: 

In [14]:
protocol.home()

#### Important Commands Runnable Separately:

In [32]:
#drop pipette in OT-2 existing trash
left.drop_tip()

<InstrumentContext: p20_single_v2.2 in LEFT>

In [29]:
#drop drySides tip in defined trash
throw = trash.wells()[1]
left.drop_tip(throw)

<InstrumentContext: p20_single_v2.2 in LEFT>

Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 402, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 415, in read_labware_latch
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 154, in get_labware_latch_status
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: 

### <b>Blink-mimicking</b> Contact Lens Wetting

<span style="color: blue;"><b>CHECKLIST:</b><br>
- is the camera on?<br>
- are the drypads placed in the correct orientation?<br>
- does user place lens first?<br>
- set <code style="color: blue;">soakTime</code> to desired value!<br>
- check # of incubations/runs (<code style="color: blue;">incubationCycleNum</code>) and # of sessile drops (<code style="color: blue;">dropletNum</code>)</span>

#### Remember to Activate Camera in Anaconda Prompt
python CameraServer.py

In [43]:
#Initiate Camera Connection
from time import time
IP_ADDRESS = "192.168.0.111"
PORT = 1200
cam = Camera(IP_ADDRESS, PORT)

In [10]:
#Parameters
userPlaceLensFirst = True                           #True if user places the lens manually first
initial_dry_time = 1.5*seconds                      #Blink protocol calls for 1.5 minutes of initial drying (@ 37°C)

#Protocol delay definitions
OT2initial_autoRun1_airTime = 76.5                  #Amount of air time before droplet for first Run1 with userPlaceLens = False

#Loop each cycle (drypad location is incubation cycle dependent)
incubationCycleNum = 10                              #Number of time lens is to be incubated: 5 times
soakTime = 5                                        ##in s; amount of time to incubate in solution; is 12 s b/c 7.28:1 ratio
dropletNum = 1                                      #Total sessile drop times; should be 1 time for Blinking

In [11]:
#Pre-Experiment Setup
gearPickUp()

Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 401, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 412, in read_rpm
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 141, in get_rpm
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response to 'M123 
'


In [18]:
a_dry = 0

Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 400, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 409, in read_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 120, in get_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response

In [15]:
##SO DON'T NEED TO MOVE THE PIPETTES MANUALLY
#Initialize Specific Variables
b = 0

#Create Folder Directory for this Trial in Desktop
run_dir = '../20260730-stenfilcon A (2024)-4p1.5m-RV2Blink-37-PBSd-MQs (2) (Showcase with BP start)/' #PBSd-50vv%ED(U)dip-PBSs
print("Created File Directory")
cam.new_folder(run_dir)

#Looping through the Incubation Cycles
for a in range(incubationCycleNum):
    if (userPlaceLensFirst and a == 0):
        #run initial drying protocol
        drySides(a_dry)
        a_dry += 1
        if (a_dry >= 96):
            a_dry = 0

        #delay for remaining time
        protocol.delay(initial_dry_time - drySides_time - droplet_time) #is 90 - 54 - 17 = 19 s
        #protocol.delay(120) #test add 20s for 120s of air time on heated pad, 173s total air time

    else:
        #run Pickup/Dropoff Lens protocol: well -> heated plate
        substratePickUp()
        flip()
        if a == 0: #if first cycle
            lensPickUp('D1')
            flip()
            protocol.delay(initial_dry_time - OT2initial_autoRun1_airTime) #is 90 - 76.5 = 13.5 s, after flip() so substrate it upright (prevents lens from falling)
        else: #o/w, soak lens first before next cycle
            #lensSoak('D1', soakTime)
            lensSoak_andPickUp('D1', soakTime-1) #-1 because takes 1 second to dip
            flip()
        substrateDropOff()

        #run initial drying protocol
        drySides(a_dry)
        a_dry += 1
        if (a_dry >= 96):
            a_dry = 0

        #delay for enough time on heated mount
        protocol.delay(OT2middle_missingHeatedPlate_time) #is 15.5 s

    ## Contact Angle Measurement
    #place droplet of MQ (or droplet solution) on contact lens
    droplet(0, a, dropletNum, run_dir, True) #dropletNum = 1 so doesn't need to be in a loop and i = 0
#     a_drop += 1
#     if (a_drop >= 48):
#         a_drop = 0

    #dry droplet from contact lens
    dryDroplet(b)

    #adjust drypad pickup to next-in-line (for next cycle)
    b += 1
    if (b >= 12):
        b = 0
        
#move lens back to well from heated plate
substratePickUp()   
flip()
lensDropOff('D1')
flip()
substrateDropOff()

print(f'a_dry: {a_dry}')

#run image analysis
print("Running Image Analysis")
cam.analyze_images(run_dir)

Created File Directory


Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 400, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 409, in read_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 120, in get_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response

Delaying for 4 seconds.


Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 400, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 409, in read_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 120, in get_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response

KeyboardInterrupt: 

In [19]:
#Post-Experiment
gearDropOff()

In [20]:
protocol.home()

Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 400, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 409, in read_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 120, in get_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response

#### Important Commands Runnable Separately:

In [16]:
#drop pipette in OT-2 existing trash
left.drop_tip()

Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 400, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 409, in read_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 120, in get_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response

<InstrumentContext: p20_single_v2.2 in LEFT>

Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 400, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 409, in read_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 120, in get_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response

In [16]:
#drop drySides tip in defined trash
throw = trash.wells()[1]
left.drop_tip(throw)

Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/utils.py", line 62, in parse_temperature_response
KeyError: 'T'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 400, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 409, in read_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 123, in get_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/utils.py", line 65, in parse_temperature_response
opentrons.drivers.utils.ParseError: ParseError(error_message=Unexpected argument to parse_temperature_response, parse_source=S:IDLE_CLOSED)


<InstrumentContext: p20_single_v2.2 in LEFT>

### Single Droplet Wetting Assay

<span style="color: blue;"><b>CHECKLIST:</b><br>
- is the camera on?<br>
- are the drypads placed in the correct orientation?<br>
- does user place lens first?<br>
- set <code style="color: blue;">soakTime</code> to desired value!<br>
- check # of incubations/runs (<code style="color: blue;">incubationCycleNum</code>) and # of sessile drops (<code style="color: blue;">dropletNum</code>)</span>

#### Remember to Activate Camera in Anaconda Prompt
python CameraServer.py

In [ ]:
#Initiate Camera Connection
from time import time
IP_ADDRESS = "192.168.0.111"
PORT = 1200
cam = Camera(IP_ADDRESS, PORT)

In [ ]:
#Parameters
userPlaceLensFirst = True                           #True if user places the lens manually first
initial_dry_time = 1.5*seconds                      #Blink protocol calls for 1.5 minutes of initial drying (@ 37°C)

#Protocol delay definitions
OT2initial_autoRun1_airTime = 76.5                  #Amount of air time before droplet for first Run1 with userPlaceLens = False

In [ ]:
#Pre-Experiment Setup
gearPickUp()

In [18]:
a_dry = 0
a_drop = 10

Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 400, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 409, in read_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 120, in get_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response

In [32]:
#Create Folder Directory for this Trial in Desktop
run_dir = '../20260710-senofilcon AOH (2024)-4p1.5m-RV2SingleDrop-37-1vv%ED(U)d (1)/' #1vv%ED(U)d
print("Created File Directory")
cam.new_folder(run_dir)

for i in range(3):
    #run initial drying protocol
    drySides(a_dry)
    a_dry += 1
    if (a_dry >= 96):
        a_dry = 0

    #delay for remaining time
    protocol.delay(initial_dry_time - drySides_time - droplet_time) #is 90 - 54 - 17 = 19 s
    #protocol.delay(120) #test add 20s for 120s of air time on heated pad, 173s total air time

    ## Contact Angle Measurement
    #place droplet of MQ (or droplet solution) on contact lens
    well_loc = 'C3'
    single_droplet(0, a_drop, i, dropletNum, run_dir, well_loc) #dropletNum = 1 so doesn't need to be in a loop and i = 0
    a_drop += 1
    if (a_drop >= 48):
        a_drop = 0
    
    protocol.delay(15) #delay 15s to remove lens and get ready to place lens
    print("Please put a new lens on within 5 sec.")
    protocol.delay(5) #delay 5s to place lens

Created File Directory


Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 400, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 409, in read_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 120, in get_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response

Please put a new lens on within 5 sec.


Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 401, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 412, in read_rpm
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 141, in get_rpm
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response to 'M123 
'
Pol

Please put a new lens on within 5 sec.


Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 400, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 409, in read_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 120, in get_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response

Please put a new lens on within 5 sec.


Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 400, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 409, in read_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 120, in get_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response

In [ ]:
#Post-Experiment
gearDropOff()

In [30]:
protocol.home()

Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 401, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 412, in read_rpm
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 141, in get_rpm
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response to 'M123 
'


#### Important Commands Runnable Separately:

In [ ]:
#drop pipette in OT-2 existing trash
left.drop_tip()

In [25]:
#drop drySides tip in defined trash
throw = trash.wells()[1]
left.drop_tip(throw)

Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 400, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 409, in read_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 120, in get_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response

<InstrumentContext: p20_single_v2.2 in LEFT>

Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 401, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 412, in read_rpm
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 141, in get_rpm
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response to 'M123 
'
Pol

### <b>Blink & Rewett</b> Contact Lens Wetting Assay

<span style="color: blue;"><b>CHECKLIST:</b><br>
- is the camera on?<br>
- are the drypads placed in the correct orientation?<br>
- does user place lens first?<br>
- set <code style="color: blue;">soakTime</code> to desired value!<br>
- check # of incubations/runs (<code style="color: blue;">incubationCycleNum</code>) and # of sessile drops (<code style="color: blue;">dropletNum</code>)</span>

#### Remember to Activate Camera in Anaconda Prompt
python CameraServer.py

In [ ]:
#Initiate Camera Connection
from time import time
IP_ADDRESS = "192.168.0.111"
PORT = 1200
cam = Camera(IP_ADDRESS, PORT)

In [ ]:
#Parameters
userPlaceLensFirst = True                           #True if user places the lens manually first

#Loop each cycle (drypad location is incubation cycle dependent)
incubationCycleNum = 5                              #Number of time lens is to be incubated: 5 times
soakTime = 94                                      #in s; amount of time to incubate in solution; 188s (8h wear), 94s (12h wear), 47s (16h wear)
dropletNum = 5                                      #Total sessile drop times; should be 1 time for Blinking

In [ ]:
#Pre-Experiment Setup
gearPickUp()

In [ ]:
'''
FIX FIRSTTTT, protocol.delay()'s are wrong
'''

#Initialize Specific Variables
b = 0

#Create Folder Directory for this Trial in Desktop
run_dir = '../20260526-senofilcon AOH (2024)-4p1.5m-RV2BR-37-MQd-OFs-94 (1)/'
print("Created File Directory")
cam.new_folder(run_dir)

#Looping through the Incubation Cycles
for a in range(incubationCycleNum):
    if (userPlaceLensFirst and a == 0):
        #run initial drying protocol
        drySides(a)

        #delay for remaining time
        protocol.delay(initial_dry_time - drySides_time) #is 90 - 71 = 19 s 

    else:
        #run Pickup/Dropoff Lens protocol: well -> heated plate
        substratePickUp()
        flip()
        if a == 0: #if first cycle
            lensPickUp('D1')
        else: #o/w, soak lens first before next cycle
            lensSoak('D1', soakTime)
        flip()
        substrateDropOff()
        protocol.delay(5)

        #run initial drying protocol
        drySides(a) 

        #no delay needed, it would have been:
        protocol.delay(OT2middle_missingHeatedPlate_time) #is 20.5 s
    
    ## Contact Angle Measurement
    for i in range(dropletNum):
        #place droplet of MQ (or droplet solution) on contact lens
        droplet(i, a, dropletNum, run_dir, False)

        #dry droplet from contact lens
        dryDroplet(b)

        #adjust drypad pickup to next-in-line (for next cycle)
        b += 1
        if (b >= 12):
            b = 0
        
#run Pickup/Dropoff Lens protocol: heated plate -> well
substratePickUp()   
flip()
lensDropOff('D1')
flip()
substrateDropOff()
    
#run image analysis
print("Running Image Analysis")
cam.analyze_images(run_dir)

In [ ]:
#Post-Experiment
gearDropOff()

In [ ]:
protocol.home()

#### Important Commands Runnable Separately:

In [ ]:
#drop pipette in OT-2 existing trash
left.drop_tip()

In [ ]:
#drop drySides tip in defined trash
throw = trash.wells()[1]
left.drop_tip(throw)

<center><h2>* * * End * * *</h2></center>

<hr style="height: 1px; border: none; background-color: gray;">

In [ ]:
#Pre-Experiment Setup
gearPickUp()

In [ ]:
#Post-Experiment
gearDropOff()

In [ ]:
protocol.home()

### Dry Tips, Droplet, Dry Pad Testing

In [ ]:
left.drop_tip()

In [ ]:
a = 0
drySides(a)

In [ ]:
#a = 0

In [ ]:
drySides(a)
a += 1

In [ ]:
#drop tip in defined trash
throw = trash.wells()[1]
left.drop_tip(throw)

In [ ]:
def droplet(i, NumDrop, CycleNum):
    stage.set_offset(x=62.5, y=0, z=-10.00)
    stg = stage.wells()[1]
    
    #Locations 
    locA = stg.top(74.5)
    locB = stg.top(72.5)

    #ASPIRATE
    pip = tips.wells()[i+NumDrop*CycleNum]
    left.pick_up_tip(pip)
    
    #Pre-lens Image
    #fname = f"../pre_lens_" + str(CycleNum) + "_" + str(i) + ".jpg"
    #cam.take_image(fname)
    
    left.aspirate(5, well['D6'])

    # CREATE DROPLET ON PIPETTE TIP     
    left.move_to(locA, force_direct = False)
    protocol.delay(1)
    left.dispense(5, locA, rate = 1) # Dispense 5 uL
    left.move_to(locB, force_direct = True, speed = 0.5)
    
    #Safety Factor
    left.move_to(locA, force_direct = True, speed = 2)
    
    #VIEW DROPLET + TAKE IMAGE   
    protocol.delay(5)
    #fname = f"../Image_" + str(CycleNum) + "_" + str(i) + ".jpg"
    #cam.take_image(fname)
    protocol.delay(1)
    
    #Drop the tip in the TRASH
    left.drop_tip() 

In [ ]:
i = 1
a = 0

# run_dir = '../TESTTTT/'
# print("Created File Directory")
# cam.new_folder(run_dir)

droplet(i, a, dropletNum)

In [ ]:
i = 0
a = 0

run_dir = '../HELLO/'
droplet(0, a, dropletNum, run_dir, True)

In [ ]:
def dryDroplet(wellNum): 
    #Remove CA droplet from contact lens using KimWipe

    #pick up drying apparatus
    loc = drypad.wells()[wellNum]
    left.pick_up_tip(loc)

    #dry the actual droplet
    stage.set_offset(x=62.5, y=0.25, z=-10.00)
    loc = stage.wells_by_name()['B1']
    
    left.move_to(loc.top(100), force_direct = False)
    left.move_to(loc.top(72), force_direct = False, speed = 70)
    left.move_to(loc.top(69.5), force_direct = False, speed = 2) #47.6, 61.85
    protocol.delay(3)
    left.move_to(loc.top(100), force_direct = False)

    #return drying apparatus
    left.return_tip()

In [ ]:
b = 0
dryDroplet(b)

In [ ]:
for b in range(12):
    dryDroplet(b)

### Pickup-Dropoff Testing

Separated Functions

In [ ]:
#Pre-Experiment Setup
gearPickUp()

In [ ]:
substratePickUp()

In [ ]:
#position offset testing for substratePickUp()
loc = stage.wells_by_name()['B2']

#move to correct z for substrate holder extension
stage.set_offset(x=-0.75, y=-62, z=-0.25)
right.move_to(loc.top(70), force_direct = False, speed = 60)
right.move_to(loc.top(32), force_direct = False, speed = 30)

#move laterally (towards) to pick up substrate holder
stage.set_offset(x=-0.75, y=-50, z=-0.25) #somehow worked at y = -49 before
right.move_to(loc.top(32), force_direct = False, speed = 5)
right.move_to(loc.top(70), force_direct = False, speed = 20)

<span style="color: red;"><b>Issue observed:</b> The part on the pinion connecting to the pickup tool may rotate and hit into the tool instead of connecting into the tool\
<b>Solution:</b> Always have the pinion in the position such that it is more stable. One side is more stable than the other side.</span>


In [ ]:
flip()

In [ ]:
lensPickUp('D1')

In [ ]:
#position offset testing for lensPickup()
lensLoc = 'D1'
loc = rack_lens.wells_by_name()[lensLoc] #Can be 'D1', D2, D3, D4, D5, D6

rack_lens.set_offset(x=0, y=-49.5, z=0)
right.move_to(loc.top(75), force_direct = False, speed = 70)
right.move_to(loc.top(10), force_direct = False, speed = 70)
right.move_to(loc.top(-6), force_direct = False, speed = 5)
protocol.delay(1.5)
right.move_to(loc.top(10), force_direct = False, speed = 5)
right.move_to(loc.top(75), force_direct = False, speed = 70)

In [ ]:
flip()

In [ ]:
substrateDropOff()

In [ ]:
#position offset testing for substrateDropOff()
loc = stage.wells_by_name()['B2']

stage.set_offset(x=-0.75, y=-50, z=-0.25) #somehow worked at y = -49 before
right.move_to(loc.top(70), force_direct = False, speed = 70)
right.move_to(loc.top(32), force_direct = False, speed = 30)

stage.set_offset(x=-0.75, y=-62, z=-0.25)
right.move_to(loc.top(32), force_direct = False, speed = 20)

In [ ]:
flip()

In [ ]:
lensDropOff('D1')

In [ ]:
#position offset testing for lensDropOff()
lensLoc = 'D1'
loc =  rack_lens.wells_by_name()[lensLoc] #Can be D1, D2, D3, D4, D5, D6

#move to lens well
rack_lens.set_offset(x=0, y=-49.5, z=0) #from lensDropOff()
right.move_to(loc.top(75), force_direct = False, speed = 70)
right.move_to(loc.top(10), force_direct = False, speed = 70)

#dip lens mount into well
right.move_to(loc.top(-2), force_direct = False, speed = 5)
protocol.delay(1.5)

#dislodge lens from well by moving in y axis
rack_lens.set_offset(x=0, y=-47, z=0) #from -2.5 from before
right.move_to(loc.top(-2), force_direct = False, speed = 5)
protocol.delay(1.5)

#lift lens mount out of well
right.move_to(loc.top(10), force_direct = False, speed = 50)
right.move_to(loc.top(75), force_direct = False, speed = 70)

In [ ]:
#Post-Experiment 
gearDropOff()

Functions Altogether

In [ ]:
#Pre-Experiment Setup
gearPickUp() #1

In [ ]:
substratePickUp()
flip()
lensPickUp('D1')
flip()
substrateDropOff()

In [ ]:
substratePickUp()
flip()
lensDropOff('D1')
flip()
substrateDropOff()

In [ ]:
substratePickUp()
flip()
lensSoak('D1', 8)
flip()
substrateDropOff()

In [ ]:
#Post-Experiment 
gearDropOff()

#### DrySides() Bending onto Lens Testing

In [ ]:
run_dir = '../HELLO/'
droplet(0, a1, dropletNum, run_dir, True)
i += 1
a1 += 1

In [ ]:
#a = 0

In [ ]:
a = 0
i = 2
a1 = 2

In [ ]:
b = 2

In [ ]:
a += 1
drySides(a)
a += 1

protocol.delay(21)

run_dir = '../HELLO/'
droplet(0, a1, dropletNum, run_dir, True)
i += 1
a1 += 1

dryDroplet(b)
# b += 1

substratePickUp()
flip()
lensSoak('D1', 10)
flip()
substrateDropOff()

drySides(a)
a += 1

run_dir = '../HELLO/'
droplet(0, a1, dropletNum, run_dir, True)
i += 1
a1 += 1

dryDroplet(b)
# b += 1

In [ ]:
substratePickUp()
flip()
lensSoak('D1', 10)
flip()
substrateDropOff()

drySides(a)
a += 1

run_dir = '../HELLO/'
droplet(0, a1, dropletNum, run_dir, True)
i += 1
a1 += 1

dryDroplet(b)
# b += 1

In [ ]:
b = 6
dryDroplet(b)
# b += 1